In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_id = "aliemre2023/berturk-financial-sentiment-analysis"
save_path = "../models/berturk-financial-sentiment-analysis"

# 1. Modeli ve Tokenizer'ı hafızaya al
print("Model indiriliyor...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

# 2. Fiziksel olarak klasöre kaydet
tokenizer.save_pretrained(save_path)
model.save_pretrained(save_path)

print(f"✅ Dosyalar başarıyla şuraya kaydedildi: {save_path}")

Model indiriliyor...


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will b

✅ Dosyalar başarıyla şuraya kaydedildi: ../models/berturk-financial-sentiment-analysis


In [ ]:
from transformers import pipeline

# yerel klasörden
yerel_yol = "../models/berturk-financial-sentiment-analysis"

analyzer = pipeline(
    "sentiment-analysis", 
    model=yerel_yol, 
    tokenizer=yerel_yol, 
    device=-1 
)

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.7) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [4]:
sonuc = analyzer("Akbank karlı bir anlaşma imzaladı.")
print(sonuc)

[{'label': 'LABEL_2', 'score': 0.9779236912727356}]


In [19]:
import re
ppath = "/Users/aliemre2023/Desktop/apache_project/bist100_airflow/src/scraper/news/2026.02.28/cointelegraph.txt"

with open(ppath, "r", encoding="utf-8") as f:
    content = f.read()

content = content.split("__________________________________________________")[2]
sentences = re.split(r'(?<![0-9])\.(?![0-9])|\n', content)
content_array = [s.strip() for s in sentences if s and len(s.strip()) > 2]


In [ ]:

import pandas as pd

# Model etiketlerini eşleyelim (Senin eğitim sırasına göre)
label_map = {
    "LABEL_0": "🔴 Negatif",
    "LABEL_1": "🟡 Nötr",
    "LABEL_2": "🟢 Pozitif"
}

# Sonuçları toplamak için bir liste
data_results = []

for line in content_array:
    clean_line = line.strip()
    if len(clean_line) > 10:  # Çok kısa (anlamsız) parçaları atla
        # Model tahmini
        prediction = analyzer(clean_line)[0]
        
        # Etiketi ve skoru al
        label = label_map.get(prediction['label'], prediction['label'])
        score = f"%{prediction['score'] * 100:.2f}"
        
        # Cümle çok uzunsa kırpalım (Crop)
        display_text = clean_line[:75] + "..." if len(clean_line) > 75 else clean_line
        
        data_results.append({
            "Cümle": display_text,
            "Duygu": label,
            "Güven Skoru": score
        })

# 2. Tabloyu Oluştur ve Yazdır
df = pd.DataFrame(data_results)

# Jupyter/Colab üzerindeysen tabloyu güzel basar, terminaldeysen standart tablo basar
print(df.to_markdown(index=False)) 

| Cümle                                                                          | Duygu      | Güven Skoru   |
|:-------------------------------------------------------------------------------|:-----------|:--------------|
| Polymarket kullanıcısı ZachXBT soruşturması üzerinden 400.000 dolar kazandı    | 🟢 Pozitif | %95.94        |
| Polymarket kullanıcıları, ZachXBT’nin içeriden bilgi alarak işlem yapma sor... | 🟢 Pozitif | %84.47        |
| ZachXBT perşembe günü X’te yaptığı paylaşımda, Axiom çalışanı Broox Bauer v... | 🟡 Nötr    | %97.23        |
| Takma adlı zincir üstü araştırmacıya göre Bauer, özel cüzdan faaliyetlerini... | 🟡 Nötr    | %97.51        |
| ZachXBT, soruşturmayla ilgili ses kayıtları paylaştı ve Bauer olduğunu söyl... | 🟡 Nötr    | %88.42        |
| Duyurunun ardından X’te yapılan paylaşımda Axiom, haber karşısında “şok ve ... | 🔴 Negatif | %48.60        |
| Axiom, “Bu araçlara erişimi kaldırdık ve soruşturmaya devam ederek sorumlu ... | 🟡 Nötr    | %77.03        |

In [ ]:
total_score = 0
sentence_count = 0

# Katsayı haritası
weights = {
    "LABEL_2": 1,  # Pozitif
    "LABEL_1": 0,  # Nötr
    "LABEL_0": -1  # Negatif
}

for line in content_array:
    clean_line = line.strip()
    if len(clean_line) > 15: # Kısa parçaları analiz dışı bırakıyoruz
        prediction = analyzer(clean_line)[0]
        label = prediction['label']
        conf_score = prediction['score']
        
        # Ağırlıklı puan hesapla: Katsayı * Güven
        current_puan = weights.get(label, 0) * conf_score

        total_score += current_puan

        if weights.get(label, 0) != 0:
            sentence_count += 1

# Ortalama Skoru Hesapla
if sentence_count > 0:
    final_sentiment_score = total_score / sentence_count
else:
    final_sentiment_score = 0

# Karar Mekanizması
if final_sentiment_score > 0.15:
    overall_sentiment = "🟢 GÜÇLÜ POZİTİF"
elif final_sentiment_score > 0.05:
    overall_sentiment = "🌿 HAFİF POZİTİF"
elif final_sentiment_score < -0.15:
    overall_sentiment = "🔴 GÜÇLÜ NEGATİF"
elif final_sentiment_score < -0.05:
    overall_sentiment = "🍂 HAFİF NEGATİF"
else:
    overall_sentiment = "⚪ NÖTR / BELİRSİZ"

print(f"--- ANALİZ ÖZETİ ---")
print(f"İşlenen Nötr Olmayan Cümle Sayısı: {sentence_count}")
print(f"Net Duygu Skoru: {final_sentiment_score:.4f}")
print(f"Genel Karar: {overall_sentiment}")

--- ANALİZ ÖZETİ ---
İşlenen Nötr Olmayan Cümle Sayısı: 6
Net Duygu Skoru: 0.4256
Genel Karar: 🟢 GÜÇLÜ POZİTİF
